# Grover's search algorithm

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/official-dvl/zksf/blob/main/examples/tutorials/grover-search.ipynb)

Find a marked item in an unsorted space faster than checking every entry.

Nothing in this notebook costs anything or needs an account until the last section. The result shown is read from the public certificate for the run that actually produced it.

Full write-up: [Grover's search algorithm](https://zksf.org/blog/grover-search-algorithm-tutorial/)


## The idea

Classically, finding one marked item among N takes about N/2 guesses. Grover's algorithm needs about the square root of N.

It works in two steps, repeated. An **oracle** flips the phase of the marked state, marking it invisibly. Then a **diffusion** step reflects every amplitude about the average, which converts that hidden phase into real amplitude. Each round pushes probability toward the answer.

Here the space is four items on two qubits and the target is `11`. At this size a single iteration is exact, so the algorithm finds it every time.


## The circuit


In [ ]:
!pip install -q qiskit


In [ ]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(2, 2)
qc.h([0, 1])         # every item equally likely

qc.cz(0, 1)          # oracle: flip the phase of |11>

qc.h([0, 1])         # diffusion: reflect about the average
qc.x([0, 1])
qc.cz(0, 1)
qc.x([0, 1])
qc.h([0, 1])

qc.measure([0, 1], [0, 1])
print(qc.draw(output='text'))


## What happened when this ran

All Clifford again, so this ran exactly on the stabilizer engine.

The cell below reads the real certificate for that run. No account, no cost.


In [ ]:
import requests

CERT = "fefb534f99a04636"
c = requests.get(f"https://api.zksf.org/certify/{CERT}/json", timeout=30).json()

print(f"engine   : {c['engine']}")
print(f"method   : {c['method']}")
print(f"shots    : {c['shots']}")
if c.get('expectation') is not None:
    print(f"energy   : {c['expectation']:.6f}")
for bits, n in (c.get('top_outcomes') or []):
    print(f"  {bits}  {n}")
print(f"accuracy : {c.get('error_bound', 'exact, shot noise only')}")
print(f"verify   : {c['verify_url']}")


## Reading the result

Every one of the 1,000 shots returned `11`, the marked item. On two qubits a single Grover iteration is exact, so there is no scatter at all. The algorithm finds the needle on the first try.

That certificate is public. Anyone can open the verify link, or check it programmatically without an account:

```
pip install zcc-verify
zcc-verify fefb534f99a04636
```


## Run it yourself

Optional, and this part does cost. Circuits this small are a fraction of a cent, and `estimate()` prices any job for free before you commit to it. Get a token from [app.zksf.org](https://app.zksf.org).


In [ ]:
!pip install -q qsim-sdk


In [ ]:
import getpass
import qsim_sdk

client = qsim_sdk.Client(token=getpass.getpass("ZKSF API token: "))

est = client.estimate(qc, shots=1000)
print('engine:', est['engine'], '| cost: $', est['predicted_cost_usd'])


In [ ]:
job = client.run(qc, shots=1000)

print(job['result'].get('counts'))
print(job['result'].get('expectation'))
print(job['result']['error_info'])


## Next

- [The full article](https://zksf.org/blog/grover-search-algorithm-tutorial/), with the maths and the background
- [All tutorials](https://zksf.org/blog/) and the [glossary](https://zksf.org/glossary-of-essential-quantum-computing-terms-for-beginners/)
- [How the accuracy statements work](https://zksf.org/quantum-computing-certification/), and [the paper](https://doi.org/10.5281/zenodo.21851381)
- [Quickstart notebook](https://colab.research.google.com/github/official-dvl/zksf/blob/main/examples/quickstart.ipynb): four certified runs, including one on real quantum hardware
